In [1]:
import os
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.ensemble import AdaBoostClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.feature_selection import RFECV, SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score, recall_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight

from lightgbm import LGBMClassifier
from mrmr import mrmr_classif
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
X_train = pd.read_excel("Data/X_train.xlsx").iloc[:, 1:]
y_train = pd.read_excel("Data/y_train.xlsx").iloc[:, 1].values
X_test  = pd.read_excel("Data/X_test.xlsx").iloc[:, 1:]
y_test  = pd.read_excel("Data/y_test.xlsx").iloc[:, 1].values

sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

In [3]:
class MRMRSelector(BaseEstimator, TransformerMixin):
    def __init__(self, K=10):
        self.K = K
        self.selected_features_ = None

    def fit(self, X, y):
        # Convert through numpy first to guarantee a clean 0-based RangeIndex on
        # both X_df and y_s. Without this, CV-sliced DataFrames keep their original
        # non-contiguous index while pd.Series(y) gets 0..n, causing mrmr's internal
        # boolean indexer to raise IndexingError on index misalignment.
        Xarr = np.array(X)
        cols = getattr(X, "columns", range(Xarr.shape[1]))
        X_df = pd.DataFrame(Xarr, columns=cols)
        y_s  = pd.Series(np.array(y).ravel())
        self.selected_features_ = mrmr_classif(X_df, y_s, K=self.K)
        return self

    def transform(self, X):
        Xarr = np.array(X)
        cols = getattr(X, "columns", range(Xarr.shape[1]))
        X_df = pd.DataFrame(Xarr, columns=cols)
        return X_df[self.selected_features_].values

In [4]:
models = {
    "XGB": XGBClassifier(eval_metric="logloss", nthread=1),
    "LR": LogisticRegression(max_iter=2000),
    "RF": RandomForestClassifier(n_jobs=1),
    "MLP": MLPClassifier(early_stopping=True),
    "SVM": SVC(probability=True, cache_size=2000),
    "AB": AdaBoostClassifier(),
    "ET": ExtraTreesClassifier(n_jobs=1),
    # min_child_samples>=5 prevents -inf gain loops on small leaves.
    # force_col_wise avoids a Windows threading deadlock in LightGBM.
    "LGBM": LGBMClassifier(verbose=-1, n_jobs=1, min_child_samples=5,
                   min_split_gain=1e-4, force_col_wise=True)
}

param_grids = {
    "XGB": {'model__max_depth':[2,3], 'model__eta':[0.01,0.03,0.3], 'model__n_estimators':[30,50,100], 'model__reg_lambda':[1,3,8]},
    "LR": {"model__C":[1e-4,1e-3,1e-2,0.1,1,10]},
    "RF": {'model__max_depth':[2,3], 'model__min_samples_leaf':[2,3,4], 'model__min_samples_split':[2,3,4], 'model__n_estimators':[50,100]},
    "MLP": {"model__hidden_layer_sizes":[(10,)], "model__alpha": [0.001,0.01,0.1,1], 'model__max_iter':[2000]},
    "SVM": {"model__C":[1e-3,0.01,0.1,1], 'model__kernel':['rbf','linear'], 'model__gamma':[0.01,0.1,1,10,100]},
    "AB": {'model__learning_rate':[0.001,0.01,0.1], 'model__estimator':[DecisionTreeClassifier(max_depth=i) for i in range(2,4)], 'model__n_estimators':[30,50,100]},
    "ET": {'model__max_depth':[2,3], 'model__min_samples_leaf':[3,4,5], 'model__min_samples_split':[2,3,4], 'model__n_estimators':[50,100]},
    "LGBM": {'model__learning_rate':[0.001,0.01,0.1,1], 'model__max_depth':[2,3], 'model__num_leaves':[5,10,20,31], 'model__n_estimators':[30,50,100]}
}

In [5]:
def get_selectors(model):
    """Return a fresh set of feature selectors for the given model.

    clone(model) gives each selector its own independent estimator so fitted
    state never leaks between the selector and the pipeline's final model.
    n_jobs=1 on all selectors prevents nested parallelism with
    RandomizedSearchCV(n_jobs=-1) which owns all cores at the outer level.
    tol=1e-3 lets sequential selectors stop early when marginal gain is tiny.
    """
    selectors = {}

    # RFECV — tree-based / gradient-boosted models only
    if isinstance(model, (RandomForestClassifier, ExtraTreesClassifier,
                          XGBClassifier, LGBMClassifier)):
        selectors["rfecv"] = RFECV(
            estimator=clone(model), step=1, cv=3,
            scoring="f1_weighted", min_features_to_select=3, n_jobs=1,
        )

    # Sequential selectors — tol stops when marginal F1 gain < 0.1 %
    selectors["ffs"] = SequentialFeatureSelector(
        clone(model), direction="forward", scoring="f1_weighted",
        cv=3, tol=1e-3, n_jobs=1,
    )
    selectors["bfs"] = SequentialFeatureSelector(
        clone(model), direction="backward", scoring="f1_weighted",
        cv=3, tol=1e-3, n_jobs=1,
    )

    selectors["mrmr"] = MRMRSelector(K=10)
    selectors["none"] = "passthrough"

    return selectors

In [6]:
def evaluate(model, X, y):
    y_pred = model.predict(X)

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X)
        if y_proba.shape[1] == 2:
            auc = roc_auc_score(y, y_proba[:,1])
        else:
            auc = roc_auc_score(y, y_proba, multi_class="ovr")
    else:
        auc = np.nan

    return {
        "f1_weighted": f1_score(y, y_pred, average="weighted"),
        "balanced_accuracy": balanced_accuracy_score(y, y_pred),
        "auc": auc,
        "sensitivity": recall_score(y, y_pred),
        "specificity": recall_score(y, y_pred, pos_label=0)
    }

In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

os.makedirs("Results", exist_ok=True)
results = []

for model_name, model in models.items():
    param_grid = param_grids[model_name]
    print(f"\nModel: {model_name}")

    for fs_name, fs in get_selectors(model).items():
        model_path = f"Results/{model_name}_{fs_name}.joblib"

        if os.path.exists(model_path):
            print(f"  FS: {fs_name}  [skipped — already exists]")
            continue

        print(f"  FS: {fs_name}")

        pipe = Pipeline([("fs", fs), ("model", model)])

        search = RandomizedSearchCV(
            pipe,
            param_distributions=param_grid,
            n_iter=20,
            cv=cv,
            scoring="f1_weighted",
            n_jobs=-1,
            random_state=42,
        )

        try:
            search.fit(X_train, y_train, model__sample_weight=sample_weights)
        except Exception:
            search.fit(X_train, y_train)
        
        # Best estimator
        best_model = search.best_estimator_

        n_feats = best_model.named_steps["model"].n_features_in_
        if fs_name != "none":
            print(f"    Number of selected features: {n_feats}")
        
        hyperp = str(search.best_params_)
        print(f"    Hyperparameters of the best estimator:\n    {hyperp}")
        
        cv_f1_mean = search.best_score_
        cv_f1_std = search.cv_results_['std_test_score'][search.best_index_]

        # Evaluation
        train_scores = evaluate(best_model, X_train, y_train)
        test_scores  = evaluate(best_model, X_test,  y_test)

        print("    f1 score:")
        print(f"     - cv: {cv_f1_mean} ({cv_f1_std})")
        print(f"     - train: {train_scores['f1_weighted']}")
        print(f"     - test: {test_scores['f1_weighted']}")

        results.append({
            "model": model_name,
            "fs": fs_name,
            "n features": n_feats,
            "hyperparameters": hyperp,
            "cv_f1_mean": cv_f1_mean,
            "cv_f1_std": cv_f1_std,
            **{f"train {k}": v for k, v in train_scores.items()},
            **{f"test {k}": v for k, v in test_scores.items()},
        })

        # Save search and model
        #joblib.dump(search, f"Results/search_{model_name}_{fs_name}.joblib")
        joblib.dump(best_model, model_path)

        print("\n")
        # Incremental save
        pd.DataFrame(results).to_excel("Results/classif_results.xlsx", index=False)

print("\nDone.")


Model: XGB
  FS: rfecv
    Number of selected features: 5
    Hyperparameters of the best estimator:
    {'model__reg_lambda': 8, 'model__n_estimators': 100, 'model__max_depth': 2, 'model__eta': 0.3}
    f1 score:
     - cv: 0.5812596780552047 (0.0625432014756915)
     - train: 0.7292611844648343
     - test: 0.5789473684210527


  FS: ffs
    Number of selected features: 4
    Hyperparameters of the best estimator:
    {'model__reg_lambda': 3, 'model__n_estimators': 50, 'model__max_depth': 3, 'model__eta': 0.01}
    f1 score:
     - cv: 0.5546261751675265 (0.044746432554985185)
     - train: 0.6760942564395577
     - test: 0.5699439055507178


  FS: bfs
    Number of selected features: 19
    Hyperparameters of the best estimator:
    {'model__reg_lambda': 3, 'model__n_estimators': 50, 'model__max_depth': 3, 'model__eta': 0.3}
    f1 score:
     - cv: 0.5601320566956138 (0.0671696413782081)
     - train: 0.917272815970485
     - test: 0.4978640776699029


  FS: mrmr


100%|██████████| 10/10 [00:04<00:00,  2.31it/s]


    Number of selected features: 10
    Hyperparameters of the best estimator:
    {'model__reg_lambda': 8, 'model__n_estimators': 100, 'model__max_depth': 2, 'model__eta': 0.3}
    f1 score:
     - cv: 0.5523695112973196 (0.05554270749954947)
     - train: 0.8094702462409423
     - test: 0.5608608592312826


  FS: none
    Hyperparameters of the best estimator:
    {'model__reg_lambda': 8, 'model__n_estimators': 100, 'model__max_depth': 2, 'model__eta': 0.3}
    f1 score:
     - cv: 0.5556133608708986 (0.03550459706796863)
     - train: 0.8796112270526596
     - test: 0.5081641623390366



Model: LR
  FS: ffs
    Number of selected features: 3
    Hyperparameters of the best estimator:
    {'model__C': 1}
    f1 score:
     - cv: 0.5883224346983464 (0.029166683792648904)
     - train: 0.6062075309108161
     - test: 0.6135575561235287


  FS: bfs
    Number of selected features: 21
    Hyperparameters of the best estimator:
    {'model__C': 0.01}
    f1 score:
     - cv: 0.56867680150

100%|██████████| 10/10 [00:04<00:00,  2.44it/s]


    Number of selected features: 10
    Hyperparameters of the best estimator:
    {'model__C': 0.1}
    f1 score:
     - cv: 0.5763042365082305 (0.0587282392827475)
     - train: 0.613887093233156
     - test: 0.5671543499205263


  FS: none
    Hyperparameters of the best estimator:
    {'model__C': 0.001}
    f1 score:
     - cv: 0.581465311156458 (0.05163182318916807)
     - train: 0.6001922869183507
     - test: 0.6571834992887624



Model: RF
  FS: rfecv
    Number of selected features: 11
    Hyperparameters of the best estimator:
    {'model__n_estimators': 50, 'model__min_samples_split': 2, 'model__min_samples_leaf': 2, 'model__max_depth': 2}
    f1 score:
     - cv: 0.5766147048375401 (0.08045502084647099)
     - train: 0.6358838728053222
     - test: 0.6050505255057613


  FS: ffs
    Number of selected features: 2
    Hyperparameters of the best estimator:
    {'model__n_estimators': 50, 'model__min_samples_split': 3, 'model__min_samples_leaf': 3, 'model__max_depth': 3}
   

...

    Number of selected features: 10
    Hyperparameters of the best estimator:
    {'model__n_estimators': 50, 'model__learning_rate': 0.1, 'model__estimator': DecisionTreeClassifier(max_depth=3)}
    f1 score:
     - cv: 0.54144156894488 (0.044100279589475974)
     - train: 0.7052447358347036
     - test: 0.6201558580935229


  FS: none
    Hyperparameters of the best estimator:
    {'model__n_estimators': 100, 'model__learning_rate': 0.1, 'model__estimator': DecisionTreeClassifier(max_depth=2)}
    f1 score:
     - cv: 0.5571157840410307 (0.04053298709796059)
     - train: 0.686647044120713
     - test: 0.5614035087719298



Model: ET
  FS: rfecv
    Number of selected features: 15
    Hyperparameters of the best estimator:
    {'model__n_estimators': 50, 'model__min_samples_split': 2, 'model__min_samples_leaf': 5, 'model__max_depth': 2}
    f1 score:
     - cv: 0.5903285678203514 (0.044428958160423214)
     - train: 0.6148067856349094
     - test: 0.625299043062201


  FS: ffs
    N

100%|██████████| 10/10 [00:02<00:00,  3.84it/s]


    Number of selected features: 10
    Hyperparameters of the best estimator:
    {'model__n_estimators': 100, 'model__min_samples_split': 3, 'model__min_samples_leaf': 4, 'model__max_depth': 2}
    f1 score:
     - cv: 0.5834817937397218 (0.054166663567380285)
     - train: 0.6237821917349025
     - test: 0.5776498256426891


  FS: none
    Hyperparameters of the best estimator:
    {'model__n_estimators': 100, 'model__min_samples_split': 2, 'model__min_samples_leaf': 5, 'model__max_depth': 2}
    f1 score:
     - cv: 0.5895852151749192 (0.062210422612579745)
     - train: 0.6283097304705345
     - test: 0.5692818324397272



Model: LGBM
  FS: rfecv
    Number of selected features: 7
    Hyperparameters of the best estimator:
    {'model__num_leaves': 10, 'model__n_estimators': 100, 'model__max_depth': 3, 'model__learning_rate': 0.1}
    f1 score:
     - cv: 0.5368830968576975 (0.07446535747105817)
     - train: 0.9172726358287977
     - test: 0.52660746694322


  FS: ffs
    Number 

100%|██████████| 10/10 [00:02<00:00,  3.89it/s]


    Number of selected features: 10
    Hyperparameters of the best estimator:
    {'model__num_leaves': 10, 'model__n_estimators': 100, 'model__max_depth': 2, 'model__learning_rate': 0.01}
    f1 score:
     - cv: 0.5566135523773811 (0.03741978483190534)
     - train: 0.6432160804020101
     - test: 0.5776498256426891


  FS: none
    Hyperparameters of the best estimator:
    {'model__num_leaves': 20, 'model__n_estimators': 100, 'model__max_depth': 3, 'model__learning_rate': 1}
    f1 score:
     - cv: 0.554431528274336 (0.07642883137688132)
     - train: 1.0
     - test: 0.4905985967082879



Done.
